# **Data Science and Machine Learning 1 - Year 4 Portfolio**
## Last.Fm API Data Pulling and Cleaning
*By Mark Lambert - C00192497*

*Lecturer: Ben O'Shaughnessey*

*Submission Date: 27th November 2026*

---

In third year I built and deployed a LastFm scrobbling program, which familiarised me with the LastFM API. 

I decided for my Data Science & Machine Learning Portfolio that using this data would be interesting and applicable.
        
This notebook focus' on connecting with the LastFm API using an API key, and then pulling scrobbles using the ```User.getRecentTracks``` and ```User.getLovedTracks``` endpoints.

The data returned requires processing and cleaning. This notebook also covers aggregating the data and merging with an external datasource to expand the features.

```%pip install pandas```

In [1]:
import requests
import pandas as pd

API_KEY = ' api key'
USERNAME = ' username'

all_tracks = []
page = 1
total_pages = 1

while page <= total_pages:
    response = requests.get('http://ws.audioscrobbler.com/2.0/', params={
        'method': 'user.getrecenttracks',
        'user': USERNAME,
        'api_key': API_KEY,
        'format': 'json',
        'limit': 200,
        'page': page
    })
    data = response.json()
    if page == 1:
        total_pages = int(data['recenttracks']['@attr']['totalPages'])
    all_tracks.extend(data['recenttracks']['track'])
    page += 1

The data's raw columns.

The ones that this portfolio will focus on are **artist, album, name and date**. ```@attr``` holds songs that are marked ```nowplaying = true```. 

As such, anything with this attribute will also be ignored later on.

In [2]:
scrobblesResponse = pd.DataFrame(all_tracks)

scrobblesResponse.columns

Index(['artist', 'streamable', 'image', 'mbid', 'album', 'name', 'url',
       'date'],
      dtype='str')

The raw data shows that the actual readable details for each artist, album are embedded with a ```#text``` key in a dictionary, which must be extracted

In [3]:
scrobblesResponse

,artist,streamable,image,mbid,album,name,url,date
0,"{'mbid': '', '#text': 'Rainbow'}",0,"[{'size': 'small', '#text': 'https://lastfm-im...",e2ee9a79-e6df-409d-9300-bd6365f9a745,{'mbid': '03157a5a-72a6-4537-b58b-ed0782c24e48...,The Temple of the King,https://www.last.fm/music/Rainbow/_/The+Temple...,"{'uts': '1792005968', '#text': '14 Oct 2026, 1..."
1,{'mbid': '6a726ac6-019e-455c-8bbb-571a77bed52e...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",12f0260c-c05a-3195-87b3-494559a3ef84,{'mbid': '2f457e5b-1a25-498f-a415-c4f9f45843bb...,Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,"{'uts': '1792005674', '#text': '14 Oct 2026, 1..."
2,"{'mbid': '', '#text': 'Alice in Chains'}",0,"[{'size': 'small', '#text': 'https://lastfm-im...",,"{'mbid': '', '#text': 'Unplugged'}","Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,"{'uts': '1792005366', '#text': '14 Oct 2026, 1..."
3,{'mbid': 'bfd085b8-0bbf-46b3-8ab9-193bca5c85e7...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",b809a9e0-411a-478c-8d46-9bfd9e195421,{'mbid': '0b5bc495-463a-4343-bb55-8ec9c60ce557...,Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,"{'uts': '1792005019', '#text': '14 Oct 2026, 1..."
4,{'mbid': '4bd95eea-b9f6-4d70-a36c-cfea77431553...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",2ba05da8-3727-3fd6-945c-1dd34757009f,{'mbid': '02244e56-c4ae-4c5c-90ab-903f94fd3ebe...,Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,"{'uts': '1792004730', '#text': '14 Oct 2026, 1..."
...,...,...,...,...,...,...,...,...
17994,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",362c85ba-c79b-478e-8321-0383c3ac8101,{'mbid': '0d2cb66c-3a32-4f81-b977-9231c461c34a...,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,"{'uts': '1763463648', '#text': '18 Nov 2025, 1..."
17995,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",6f93589f-4a47-4a9d-9274-09a0f437ad15,{'mbid': '10ac58ca-0655-4b16-a6cb-58fdc309de0a...,Over,https://www.last.fm/music/Portishead/_/Over,"{'uts': '1763463370', '#text': '18 Nov 2025, 1..."
17996,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",6da0b598-d9cf-4f62-8376-bc3181e509af,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,"{'uts': '1763462844', '#text': '18 Nov 2025, 1..."
17997,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,0,"[{'size': 'small', '#text': 'https://lastfm-im...",ad79b526-8df8-4e79-8756-0607df153e59,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,"{'uts': '1763462559', '#text': '18 Nov 2025, 1..."


Next, as per [Carlos Medina's article](https://medium.com/hemisferios/using-the-last-fm-api-to-extract-data-and-analyzing-it-with-python-part-1-ffe96f812455), first we must select the colums of interest and, as the column name is obscure I have chosen to rename it to **song**

In [4]:
scrobbles = scrobblesResponse

scrobbles = scrobbles[['artist', 'album', 'name', 'url', 'date']]
scrobbles.rename(columns={'name': 'song'}, inplace=True)
scrobbles

,artist,album,song,url,date
0,"{'mbid': '', '#text': 'Rainbow'}",{'mbid': '03157a5a-72a6-4537-b58b-ed0782c24e48...,The Temple of the King,https://www.last.fm/music/Rainbow/_/The+Temple...,"{'uts': '1792005968', '#text': '14 Oct 2026, 1..."
1,{'mbid': '6a726ac6-019e-455c-8bbb-571a77bed52e...,{'mbid': '2f457e5b-1a25-498f-a415-c4f9f45843bb...,Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,"{'uts': '1792005674', '#text': '14 Oct 2026, 1..."
2,"{'mbid': '', '#text': 'Alice in Chains'}","{'mbid': '', '#text': 'Unplugged'}","Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,"{'uts': '1792005366', '#text': '14 Oct 2026, 1..."
3,{'mbid': 'bfd085b8-0bbf-46b3-8ab9-193bca5c85e7...,{'mbid': '0b5bc495-463a-4343-bb55-8ec9c60ce557...,Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,"{'uts': '1792005019', '#text': '14 Oct 2026, 1..."
4,{'mbid': '4bd95eea-b9f6-4d70-a36c-cfea77431553...,{'mbid': '02244e56-c4ae-4c5c-90ab-903f94fd3ebe...,Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,"{'uts': '1792004730', '#text': '14 Oct 2026, 1..."
...,...,...,...,...,...
17994,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '0d2cb66c-3a32-4f81-b977-9231c461c34a...,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,"{'uts': '1763463648', '#text': '18 Nov 2025, 1..."
17995,{'mbid': '8f6bd1e4-fbe1-4f50-aa9b-94c450ec0f11...,{'mbid': '10ac58ca-0655-4b16-a6cb-58fdc309de0a...,Over,https://www.last.fm/music/Portishead/_/Over,"{'uts': '1763463370', '#text': '18 Nov 2025, 1..."
17996,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,"{'uts': '1763462844', '#text': '18 Nov 2025, 1..."
17997,{'mbid': '63aa26c3-d59b-4da4-84ac-716b54f1ef4d...,{'mbid': '51f8289e-7379-4c7b-ab50-564e2f875594...,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,"{'uts': '1763462559', '#text': '18 Nov 2025, 1..."


Next, convert the ```artist``` ```album``` and ```date``` to strings

In [5]:
scrobbles['artist'] = scrobbles['artist'].astype(str)
scrobbles['album'] = scrobbles['album'].astype(str)
scrobbles['date'] = scrobbles['date'].astype(str)

Extract the embedded ```#text``` attribute from each of the ```artist, album``` and ```date``` fields

In [6]:
scrobbles[['mbid2','artist']]=scrobbles['artist'].str.split("#text':", n = 1, expand = True)
scrobbles= scrobbles.drop('mbid2', axis=1)
scrobbles[['mbid3','album']]=scrobbles['album'].str.split("#text':", n = 1, expand = True)
scrobbles= scrobbles.drop('mbid3', axis=1)
scrobbles[['uts','date']]=scrobbles['date'].str.split("#text':", n = 1, expand = True)
scrobbles.head()

,artist,album,song,url,date,uts
0,'Rainbow'},"""Ritchie Blackmore's Rainbow""}",The Temple of the King,https://www.last.fm/music/Rainbow/_/The+Temple...,"'14 Oct 2026, 19:26'}","{'uts': '1792005968', '"
1,'Thin Lizzy'},'Bad Reputation'},Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,"'14 Oct 2026, 19:21'}","{'uts': '1792005674', '"
2,'Alice in Chains'},'Unplugged'},"Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,"'14 Oct 2026, 19:16'}","{'uts': '1792005366', '"
3,'Mad Season'},'Above'},Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,"'14 Oct 2026, 19:10'}","{'uts': '1792005019', '"
4,'Alice in Chains'},'Alice in Chains'},Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,"'14 Oct 2026, 19:05'}","{'uts': '1792004730', '"


Removing the ```}``` and ```'``` characters from the ```artist and album``` entries

In [7]:
scrobbles['artist'] = scrobbles['artist'].str.strip("}'\" ")
scrobbles['album'] = scrobbles['album'].str.strip("}'\" ")
scrobbles.head()

,artist,album,song,url,date,uts
0,Rainbow,Ritchie Blackmore's Rainbow,The Temple of the King,https://www.last.fm/music/Rainbow/_/The+Temple...,"'14 Oct 2026, 19:26'}","{'uts': '1792005968', '"
1,Thin Lizzy,Bad Reputation,Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,"'14 Oct 2026, 19:21'}","{'uts': '1792005674', '"
2,Alice in Chains,Unplugged,"Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,"'14 Oct 2026, 19:16'}","{'uts': '1792005366', '"
3,Mad Season,Above,Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,"'14 Oct 2026, 19:10'}","{'uts': '1792005019', '"
4,Alice in Chains,Alice in Chains,Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,"'14 Oct 2026, 19:05'}","{'uts': '1792004730', '"


The ```date``` and ```uts``` fields also need to be stripped of the remaining dirty characters. I have also chosen to rename uts to **timestamp** as it is more descriptive.

In [8]:
scrobbles['date'] = scrobbles['date'].str.strip("}'\" ")
scrobbles['uts'] = scrobbles['uts'].str.strip("{uts':','\" ")
scrobbles.rename(columns={'uts': 'timestamp'}, inplace=True)
scrobbles.head()

,artist,album,song,url,date,timestamp
0,Rainbow,Ritchie Blackmore's Rainbow,The Temple of the King,https://www.last.fm/music/Rainbow/_/The+Temple...,"14 Oct 2026, 19:26",1792005968
1,Thin Lizzy,Bad Reputation,Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,"14 Oct 2026, 19:21",1792005674
2,Alice in Chains,Unplugged,"Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,"14 Oct 2026, 19:16",1792005366
3,Mad Season,Above,Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,"14 Oct 2026, 19:10",1792005019
4,Alice in Chains,Alice in Chains,Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,"14 Oct 2026, 19:05",1792004730


Next, the date field must be converted into a date only field

In [9]:
scrobbles['date'] = pd.to_datetime(scrobbles['date'], unit='s')
scrobbles['date_only'] = scrobbles['date'].dt.date
scrobbles

,artist,album,song,url,date,timestamp,date_only
0,Rainbow,Ritchie Blackmore's Rainbow,The Temple of the King,https://www.last.fm/music/Rainbow/_/The+Temple...,2026-10-14 19:26:00,1792005968,2026-10-14
1,Thin Lizzy,Bad Reputation,Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,2026-10-14 19:21:00,1792005674,2026-10-14
2,Alice in Chains,Unplugged,"Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,2026-10-14 19:16:00,1792005366,2026-10-14
3,Mad Season,Above,Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,2026-10-14 19:10:00,1792005019,2026-10-14
4,Alice in Chains,Alice in Chains,Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,2026-10-14 19:05:00,1792004730,2026-10-14
...,...,...,...,...,...,...,...
17994,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18
17995,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18
17996,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18
17997,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18


Finally, the row which contains NaN must be removed, as this was a song that had the ```@attr``` of ```nowplaying = true```, so it should not count as a fully scrobbled song

In [10]:
scrobbles = scrobbles.dropna(subset=['date'])
scrobbles

,artist,album,song,url,date,timestamp,date_only
0,Rainbow,Ritchie Blackmore's Rainbow,The Temple of the King,https://www.last.fm/music/Rainbow/_/The+Temple...,2026-10-14 19:26:00,1792005968,2026-10-14
1,Thin Lizzy,Bad Reputation,Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,2026-10-14 19:21:00,1792005674,2026-10-14
2,Alice in Chains,Unplugged,"Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,2026-10-14 19:16:00,1792005366,2026-10-14
3,Mad Season,Above,Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,2026-10-14 19:10:00,1792005019,2026-10-14
4,Alice in Chains,Alice in Chains,Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,2026-10-14 19:05:00,1792004730,2026-10-14
...,...,...,...,...,...,...,...
17994,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18
17995,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18
17996,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18
17997,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18


Then, three new columns- ```hour``` ```tod``` ```month``` are added. First, the time has to be converted to Dublin's time. Then, we can call the time_of_day function to classify between the four periods of a day: **morning, afternoon, evening and night.**

In [11]:
def time_of_day(hour):
    if hour >= 22 or hour < 5:
        return 'night'
    if hour < 12:
        return 'morning'
    if hour < 17:
        return 'afternoon'
    return 'evening'

local_time = scrobbles['date'].dt.tz_localize('UTC').dt.tz_convert('Europe/Dublin')
scrobbles['hour'] = local_time.dt.hour
scrobbles['tod'] = scrobbles['hour'].apply(time_of_day)
scrobbles['month'] = scrobbles['date'].dt.to_period('M')
scrobbles.head()

,artist,album,song,url,date,timestamp,date_only,hour,tod,month
0,Rainbow,Ritchie Blackmore's Rainbow,The Temple of the King,https://www.last.fm/music/Rainbow/_/The+Temple...,2026-10-14 19:26:00,1792005968,2026-10-14,20,evening,2026-10
1,Thin Lizzy,Bad Reputation,Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,2026-10-14 19:21:00,1792005674,2026-10-14,20,evening,2026-10
2,Alice in Chains,Unplugged,"Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,2026-10-14 19:16:00,1792005366,2026-10-14,20,evening,2026-10
3,Mad Season,Above,Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,2026-10-14 19:10:00,1792005019,2026-10-14,20,evening,2026-10
4,Alice in Chains,Alice in Chains,Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,2026-10-14 19:05:00,1792004730,2026-10-14,20,evening,2026-10


Aggregate the data into daily scrobbles

In [12]:
daily = scrobbles.groupby('date_only').size().reset_index(name='scrobble_count')
daily['date_only'] = pd.to_datetime(daily['date_only'])
daily.set_index('date_only', inplace=True)

In [13]:
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
for i in range(7):
    daily[days[i]] = (daily.index.dayofweek == i).astype(float)

In [14]:
monthly = scrobbles.groupby('month').size().reset_index(name='scrobble_count')
monthly

,month,scrobble_count
0,2025-11,1254
1,2025-12,1732
2,2026-01,1692
3,2026-02,1345
4,2026-03,1022
5,2026-04,522
6,2026-05,829
7,2026-06,1942
8,2026-07,2479
9,2026-08,2430


Output the data to csv's so that I don't have to request the data each time I run the program

In [15]:
scrobbles.to_csv('datasets/scrobbles.csv', index=False)

In [16]:
daily.to_csv('datasets/daily.csv', index=True)

Using the [unofficial docs](https://lastfm-docs.github.io/api-docs/user/getLovedTracks/#examples) once again, the data can be combined with the ```User.getLovedTracks``` endpoint. A new field called ```is_loved``` can be joined to the full scrobble dataset

In [17]:
all_loved_tracks = []
loved_page = 1
total_loved_pages = 1

while loved_page <= total_loved_pages:
    response = requests.get('http://ws.audioscrobbler.com/2.0/', params={
        'method': 'user.getlovedtracks',
        'user': USERNAME,
        'api_key': API_KEY,
        'format': 'json',
        'limit': 200,
        'page': loved_page
    })
    data = response.json()
    if loved_page == 1:
        total_loved_pages = int(data['lovedtracks']['@attr']['totalPages'])
    tracks = data['lovedtracks']['track']
    for track in tracks:
        all_loved_tracks.append({
            'artist': track['artist']['name'],
            'track': track['name'],
            'timestamp': int(track['date']['uts']) if 'date' in track else None
        })
    loved_page += 1

Merging the scrobbles and retrieved lovedScrobbles tracks on the columns ```artist``` and ```song``` against ```track``` and adding the new col ```is_loved``` 

In [18]:
lovedScrobbles = pd.DataFrame(all_loved_tracks)
lovedScrobbles['is_loved'] = 1
lovedScrobbles = lovedScrobbles.drop_duplicates(subset=['artist', 'track'])

lovedMerged = pd.merge(
    scrobbles,
    lovedScrobbles[['artist', 'track', 'is_loved']],
    left_on=['artist', 'song'],
    right_on=['artist', 'track'],
    how='left'
)

loved = lovedMerged.drop(columns=['track'])
loved['is_loved'] = loved['is_loved'].fillna(0).astype(int)
loved

,artist,album,song,url,date,timestamp,date_only,hour,tod,month,is_loved
0,Rainbow,Ritchie Blackmore's Rainbow,The Temple of the King,https://www.last.fm/music/Rainbow/_/The+Temple...,2026-10-14 19:26:00,1792005968,2026-10-14,20,evening,2026-10,0
1,Thin Lizzy,Bad Reputation,Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,2026-10-14 19:21:00,1792005674,2026-10-14,20,evening,2026-10,1
2,Alice in Chains,Unplugged,"Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,2026-10-14 19:16:00,1792005366,2026-10-14,20,evening,2026-10,0
3,Mad Season,Above,Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,2026-10-14 19:10:00,1792005019,2026-10-14,20,evening,2026-10,1
4,Alice in Chains,Alice in Chains,Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,2026-10-14 19:05:00,1792004730,2026-10-14,20,evening,2026-10,1
...,...,...,...,...,...,...,...,...,...,...,...
17994,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18,11,morning,2025-11,1
17995,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18,10,morning,2025-11,1
17996,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18,10,morning,2025-11,1
17997,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18,10,morning,2025-11,0


In [19]:
loved.to_csv('datasets/loved.csv', index=False)

In [20]:
spotify = pd.read_csv('datasets/spotify.csv')

Using a Spotify track dataset from [Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset) which contains additional features such as ```danceability```, ```energy``` and ```tempo```

In order to get a higher match rate, the artist and track features are stripped and lowercase 

In [21]:
spotify['clean_artist'] = spotify['artists'].str.lower().str.strip()
spotify['clean_track'] = spotify['track_name'].str.lower().str.strip()

loved['clean_artist'] = loved['artist'].str.lower().str.strip()
loved['clean_track'] = loved['song'].str.lower().str.strip()

spotify_unique = spotify.drop_duplicates(subset=['clean_artist', 'clean_track'])

spotifyMerged = pd.merge(
    loved,
    spotify_unique,
    on=['clean_artist', 'clean_track'],
    how='left',
    indicator=True
)

matched_count = (spotifyMerged['_merge'] == 'both').sum()
total_count = len(loved)
match_rate = (matched_count / total_count) * 100

print(f"{matched_count} of {total_count} tracks match ({match_rate:.2f}%)")

spotifyMerged

6313 of 17999 tracks match (35.07%)


,artist,album,song,url,date,timestamp,date_only,hour,tod,month,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,_merge
0,Rainbow,Ritchie Blackmore's Rainbow,The Temple of the King,https://www.last.fm/music/Rainbow/_/The+Temple...,2026-10-14 19:26:00,1792005968,2026-10-14,20,evening,2026-10,...,0.0,0.0341,0.44500,0.000014,0.0648,0.373,81.972,4.0,blues,both
1,Thin Lizzy,Bad Reputation,Southbound,https://www.last.fm/music/Thin+Lizzy/_/Southbound,2026-10-14 19:21:00,1792005674,2026-10-14,20,evening,2026-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,Alice in Chains,Unplugged,"Down In A Hole - Live at the Majestic Theatre,...",https://www.last.fm/music/Alice+in+Chains/_/Do...,2026-10-14 19:16:00,1792005366,2026-10-14,20,evening,2026-10,...,1.0,0.0240,0.08830,0.000027,0.8830,0.432,95.125,4.0,grunge,both
3,Mad Season,Above,Long Gone Day,https://www.last.fm/music/Mad+Season/_/Long+Go...,2026-10-14 19:10:00,1792005019,2026-10-14,20,evening,2026-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,Alice in Chains,Alice in Chains,Heaven Beside You,https://www.last.fm/music/Alice+in+Chains/_/He...,2026-10-14 19:05:00,1792004730,2026-10-14,20,evening,2026-10,...,0.0,0.0320,0.00451,0.000167,0.1160,0.559,111.243,4.0,grunge,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17994,Tame Impala,Lonerism,Why Won't They Talk to Me?,https://www.last.fm/music/Tame+Impala/_/Why+Wo...,2025-11-18 11:00:00,1763463648,2025-11-18,11,morning,2025-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
17995,Portishead,Portishead,Over,https://www.last.fm/music/Portishead/_/Over,2025-11-18 10:56:00,1763463370,2025-11-18,10,morning,2025-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
17996,Tame Impala,Deadbeat,Obsolete,https://www.last.fm/music/Tame+Impala/_/Obsolete,2025-11-18 10:47:00,1763462844,2025-11-18,10,morning,2025-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
17997,Tame Impala,Deadbeat,Piece Of Heaven,https://www.last.fm/music/Tame+Impala/_/Piece+...,2025-11-18 10:42:00,1763462559,2025-11-18,10,morning,2025-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


We make a new dataset called ```aggregated```

First we take one row per unique track w/ the spotify features selected in ```audio_cols```. 
Then we add the ```play_count``` for each track and remove any tracks that didn't match between the two datasets

The result is one row per matched track with the audio features and play count for each track. 

In [22]:
audio_cols = ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'tempo', 'speechiness']

aggregated = spotifyMerged.groupby(['clean_artist', 'clean_track'])[audio_cols].first().reset_index()
aggregated['play_count'] = spotifyMerged.groupby(['clean_artist', 'clean_track'])['is_loved'].size().values
aggregated = aggregated.dropna(subset=['danceability', 'energy', 'valence'])
aggregated.to_csv('datasets/aggregatedData.csv', index=False)
aggregated

,clean_artist,clean_track,danceability,energy,valence,acousticness,instrumentalness,tempo,speechiness,play_count
4,a família,brinquedo assassino,0.771,0.746,0.502,0.090200,0.000000,148.054,0.2620,2
13,abba,super trouper,0.724,0.724,0.927,0.294000,0.000000,117.986,0.0304,1
14,adnan sami,tera chehra,0.646,0.562,0.488,0.523000,0.002100,100.000,0.0331,1
20,air,kelly watch the stars,0.516,0.743,0.226,0.282000,0.637000,109.837,0.0292,17
21,air,la femme d'argent,0.517,0.737,0.558,0.434000,0.684000,79.820,0.0352,20
...,...,...,...,...,...,...,...,...,...,...
2871,weezer,pork and beans,0.599,0.626,0.708,0.010800,0.000009,119.837,0.0339,1
2872,weezer,say it ain't so,0.635,0.551,0.451,0.007430,0.000018,151.779,0.0354,2
2874,weezer,the world has turned and left me here,0.455,0.918,0.560,0.000399,0.013400,99.078,0.0352,1
2875,weezer,undone - the sweater song,0.457,0.618,0.579,0.001140,0.002570,80.184,0.0296,1
